# Solutions — String Functions Basics

**Dataset:** `samples.bakehouse.sales_customers`, `samples.bakehouse.sales_transactions`, `samples.tpch.customer`

**Topics:** upper, lower, concat, split, length, substring, contains

> Reference solutions for practice notebooks.
> **Try the problem yourself first** — only open this when you've made a genuine attempt.
>
> Each solution includes a `# Why:` comment explaining the approach chosen.

In [ ]:
from pyspark.sql import functions as F, types as T
customers    = spark.table("samples.bakehouse.sales_customers")
transactions = spark.table("samples.bakehouse.sales_transactions")
tpch_customer = spark.table("samples.tpch.customer")

---
## Problem 1 — Standardise Customer Names

In [ ]:
# Why: build each derived column in separate withColumn calls so they can reference each other.
result_1 = (
    customers.withColumn("upper_first", F.upper("first_name"))
             .withColumn("lower_last",  F.lower("last_name"))
             .withColumn("full_name",   F.concat(F.col("upper_first"), F.lit(" "), F.col("lower_last")))
             .select("customerID", "full_name", "upper_first", "lower_last")
)

---
## Problem 2 — Email Username Extraction

In [ ]:
# Why: F.split() returns an ArrayType column; index with [0] and [1] to extract parts.
result_2 = (
    customers.withColumn("username", F.split("email_address", "@")[0])
             .withColumn("domain",   F.split("email_address", "@")[1])
             .select("customerID", "email_address", "username", "domain")
)

---
## Problem 3 — Name Length Analysis

In [ ]:
# Why: compute intermediate columns first, then filter on the derived total_length.
result_3 = (
    customers.withColumn("first_name_length", F.length("first_name"))
             .withColumn("last_name_length",  F.length("last_name"))
             .withColumn("total_length", F.col("first_name_length") + F.col("last_name_length"))
             .filter(F.col("total_length") > 20)
             .select("customerID", "first_name", "last_name", "first_name_length", "last_name_length", "total_length")
)

---
## Problem 4 — Product Name Search

In [ ]:
# Why: .contains() is clean and readable; .like("%Coffee%") is an equivalent alternative.
result_4 = (
    transactions.filter(F.col("product").contains("Coffee"))
                .groupBy("product")
                .count()
                .withColumnRenamed("count", "transaction_count")
)

---
## Problem 5 — Substring Extraction

In [ ]:
# Why: F.substring uses 1-based indexing — position 1 is the first character.
result_5 = (
    tpch_customer.withColumn("country_code", F.substring("c_phone", 1, 3))
                 .groupBy("country_code")
                 .count()
                 .withColumnRenamed("count", "customer_count")
                 .orderBy("country_code")
)